# Week 5 — Multi-Agent Systems & State Management

> **Source notebook for** [`src/multi_agent/`](../src/multi_agent/).

## Learning objectives

1. Compare multi-agent topologies (hierarchical, peer-to-peer, blackboard) and choose the right one for a task.
2. Model shared state as a structured blackboard and implement loop detection via content hashing.
3. Build a self-correcting Coder–Executor–Critic system in which one agent writes code, another runs it, and a third diagnoses failures.
4. Prove termination of the loop and articulate why naive turn-counting is insufficient.
5. Distinguish episodic from semantic memory and explain where each belongs in an agent system.


## 1. Why more than one agent?

A single ReAct agent must hold the entire problem in one context and one persona. Multi-agent systems decompose a problem across *specialized* agents, each with a focused prompt and a narrow responsibility. The benefits:

- **Separation of concerns.** A "Coder" prompt optimized for writing code is different from a "Critic" prompt optimized for finding bugs. One agent cannot be optimal at both.
- **Adversarial robustness.** A Critic that did not write the code has no sunk-cost bias toward defending it. This is the multi-agent analogue of code review.
- **Bounded context.** Each agent sees only what it needs, keeping individual contexts small and focused.

The cost is **coordination complexity**: state must be shared, turns must be sequenced, and the system must be *guaranteed to terminate*.


## 2. Topologies

| Topology | Structure | When to use |
|----------|-----------|-------------|
| **Hierarchical** | A manager agent delegates to worker agents and aggregates results | Tasks with clear decomposition (research → sections → synthesis) |
| **Peer-to-peer** | Agents converse directly, no central authority | Negotiation, debate, consensus-seeking |
| **Blackboard** | Agents read/write a shared state object; a controller sequences them | Iterative refinement with a shared artifact (our case) |

Our self-correcting coder uses the **blackboard** pattern: a shared `AgentState` object holds the current code, the last error, and the conversation history. A controller (`SelfCorrectingCoder`) sequences the Coder, Executor, and Critic.


## 3. Shared state and loop detection

The single hardest problem in multi-agent systems is the **non-terminating loop**: two agents oscillating between two positions, or repeatedly producing the same broken artifact.

Naive turn-counting (`max_iterations`) bounds the worst case but wastes budget on cycles. A better guard: **hash the productive content of the state each round; halt if a hash repeats.**

The state lives in [`src/multi_agent/state.py`](../src/multi_agent/state.py).


In [ ]:
from src.multi_agent.state import AgentState

state = AgentState(task="implement merge sort")

# Round 1: the Coder writes some code that fails.
state.code = "def merge_sort(x): return sorted(x)  # placeholder"
state.last_error = "AssertionError: stability test failed"
print("loop on first sight of this state?", state.detect_loop())   # False — new

# Round 2: the Coder produces the *exact same* code and the *exact same* error.
print("loop on repeat?                  ", state.detect_loop())   # True — stalled


In [ ]:
# But genuine progress resets the detector.
state2 = AgentState(task="t")
state2.code = "v1"; state2.last_error = "err A"
print("v1/errA  ->", state2.detect_loop())   # False

state2.code = "v2"; state2.last_error = "err B"   # the Coder revised → progress
print("v2/errB  ->", state2.detect_loop())   # False


The fingerprint hashes `(code, last_error)` because those are the *productive* components of the state. Conversation history grows monotonically and would never repeat, so hashing it would defeat the detector. Choosing what to hash is a design decision: hash the artifact whose stagnation defines a stalled loop.


## 4. The Coder–Executor–Critic topology

```
   task
    │
    ▼
 ┌────────┐   code   ┌──────────┐  pass/fail+stderr  ┌──────────┐
 │ Coder  │─────────▶│ Executor │───────────────────▶│ (check)  │
 └────────┘          └──────────┘                    └────┬─────┘
    ▲                                                      │ fail
    │ revise(code, error, critique)                        ▼
    │                                                  ┌──────────┐
    └──────────────── critique ────────────────────────│  Critic  │
                                                        └──────────┘
```

- **Coder** (LLM): writes and revises Python.
- **Executor** (*not* an LLM): runs the code in a subprocess with a timeout, captures stdout/stderr/returncode.
- **Critic** (LLM): given failing code and its traceback, diagnoses the root cause and proposes a fix strategy in natural language.

The Executor being deterministic (not an LLM) is deliberate: code execution must be ground truth, not a model's *opinion* about whether code would run.


In [ ]:
# The Executor in isolation — pure subprocess execution, no LLM.
from src.multi_agent.agents import ExecutorAgent

executor = ExecutorAgent(timeout_seconds=5.0)

good = '''
def merge_sort(a):
    if len(a) <= 1: return a
    mid = len(a)//2
    L, R = merge_sort(a[:mid]), merge_sort(a[mid:])
    out, i, j = [], 0, 0
    while i < len(L) and j < len(R):
        if L[i] <= R[j]: out.append(L[i]); i += 1
        else: out.append(R[j]); j += 1
    return out + L[i:] + R[j:]

import random
data = list(range(1000)); random.shuffle(data)
assert merge_sort(data) == sorted(data)
print("OK: merge sort verified on 1000 elements")
'''

result = executor.run(good)
print("passed:    ", result.passed)
print("returncode:", result.returncode)
print("stdout:    ", result.stdout.strip())


In [ ]:
# A failing program → the Executor captures the traceback.
bad = '''
def merge_sort(a):
    return a[::-1]   # wrong: just reverses
import random
data = list(range(100)); random.shuffle(data)
assert merge_sort(data) == sorted(data), "not sorted!"
print("OK")
'''
result = executor.run(bad)
print("passed:", result.passed)
print("stderr tail:", result.stderr.strip().splitlines()[-1])


## 5. The full self-correcting loop

`SelfCorrectingCoder` ([`src/multi_agent/orchestrator.py`](../src/multi_agent/orchestrator.py)) wires the three agents together with the loop detector and the iteration budget. We demonstrate it with scripted LLMs so the notebook runs without API keys; the real version uses `build_client(...)`.


In [ ]:
from src.multi_agent.orchestrator import SelfCorrectingCoder
from src.multi_agent.agents import CoderAgent, CriticAgent, ExecutorAgent

class FakeLLM:
    def __init__(self, scripted): self.scripted = list(scripted)
    def complete(self, messages, **kw): return self.scripted.pop(0)

# Coder first writes a buggy version, then (after the critique) a correct one.
buggy = '''```python
def is_palindrome(s):
    return s == s   # bug: always True
assert is_palindrome("racecar")
assert not is_palindrome("hello"), "should reject non-palindrome"
print("OK")
```'''

fixed = '''```python
def is_palindrome(s):
    return s == s[::-1]
assert is_palindrome("racecar")
assert not is_palindrome("hello")
print("OK: palindrome checker verified")
```'''

coder_llm  = FakeLLM([buggy, fixed])              # initial(), then revise()
critic_llm = FakeLLM(["The function compares s to itself instead of its reverse. Compare s to s[::-1]."])

coder = SelfCorrectingCoder.__new__(SelfCorrectingCoder)   # build manually w/ fakes
coder.coder    = CoderAgent(coder_llm)
coder.executor = ExecutorAgent(timeout_seconds=5.0)
coder.critic   = CriticAgent(critic_llm)
coder.max_iterations = 5

result = coder.solve("Write an is_palindrome(s) function with self-checks.")
print("success:      ", result.success)
print("iterations:   ", result.iterations)
print("halted_reason:", result.halted_reason)
print("--- final code ---")
print(result.final_code)


**What happened:** iteration 1 ran the buggy code, the Executor reported the assertion failure, the Critic diagnosed the `s == s` bug, the Coder revised to `s == s[::-1]`, and iteration 2 passed. The loop terminated with `halted_reason="success"` in 2 iterations.


## 6. Termination guarantees

Let the state at iteration $t$ be $s_t = (c_t, e_t)$ — code and most-recent error. The orchestrator maintains a set $\mathcal{H}$ of seen state hashes. Two guarantees:

1. **Hard bound.** The loop runs at most `max_iterations` times regardless of behaviour.
2. **Cycle bound.** If $\text{hash}(s_t) \in \mathcal{H}$, the loop halts immediately with `halted_reason="loop_detected"`.

Together these ensure the system *always halts*, and halts *early* when it detects it is making no progress — instead of burning the full budget on a cycle.


In [ ]:
# Demonstrate loop detection: a Coder that keeps producing the same broken code.
stuck_code = '''```python
assert 1 == 2, "always fails the same way"
```'''
coder_llm  = FakeLLM([stuck_code, stuck_code, stuck_code, stuck_code])
critic_llm = FakeLLM(["The assertion 1 == 2 is impossible."] * 4)

coder = SelfCorrectingCoder.__new__(SelfCorrectingCoder)
coder.coder    = CoderAgent(coder_llm)
coder.executor = ExecutorAgent(timeout_seconds=5.0)
coder.critic   = CriticAgent(critic_llm)
coder.max_iterations = 10

result = coder.solve("impossible task")
print("halted_reason:", result.halted_reason, "after", result.iterations, "iterations")


Note the loop halted after **2** iterations, not 10 — the detector caught the repeated `(code, error)` fingerprint and aborted, saving 8 wasted LLM calls.


## 7. Memory: episodic vs. semantic

Two kinds of memory belong in an agent system:

- **Episodic memory** — the *trace* of what happened: which actions were taken, what observations returned. Our `AgentState.history` is episodic. It is short-lived (per-task) and supports loop detection and debugging.
- **Semantic memory** — *distilled knowledge* that outlives a single task: "this corpus uses 'transformer' to mean the architecture, not the electrical device." Semantic memory is typically a vector store (Week 2!) that the agent retrieves from.

A mature agent writes salient episodic events into semantic memory at the end of a task — closing the loop between the agentic and retrieval halves of this course.


## 8. Exercises

1. **Add a Planner.** Introduce a fourth agent that decomposes the task into subtasks *before* the Coder starts. Does it improve success rate on multi-part tasks?
2. **Sandbox hardening.** The Executor runs code in a subprocess, not a container. Replace it with a Docker-based executor and discuss the threat model (filesystem, network, resource exhaustion).
3. **Critic calibration.** Sometimes the Critic misdiagnoses. Add a confidence score to the Critic's output and have the Coder weight revisions accordingly. Measure on a suite of buggy programs.
4. **Persistent semantic memory.** After each solved task, embed a one-line lesson ("merge sort needs a stability test") into a Chroma store. Retrieve relevant lessons at the start of the next task.


## 9. Take-aways

- Multi-agent systems trade coordination complexity for separation of concerns and adversarial robustness. Use them when a single persona/context cannot do the job well.
- The **blackboard** pattern — shared state, sequenced agents — is the workhorse topology for iterative refinement.
- **Hash the productive state for loop detection.** Turn-counting bounds the worst case; content-hashing catches stalls early.
- The Executor must be **ground truth**, not an LLM. Code either runs or it doesn't.
- Distinguish episodic (per-task trace) from semantic (durable knowledge) memory; the latter is a vector store, connecting back to Weeks 2–3.

➡ Next week we compose everything — MCP, RAG, re-ranking, ReAct, and multi-agent state — into a single autonomous Academic Research Assistant.
